In [1]:
import pandas as pd
import os
import openai
from typing import List, Dict, Optional
import json
from datetime import datetime

import textwrap

In [ ]:
# use open ai model to generate a story about a city

    "# Get API key from environment variables
",
    "import os
",
    "from dotenv import load_dotenv
",
    "load_dotenv()
",
    "openai_key = os.getenv('OPENAI_API_KEY')
",
    "
",
    "chat_model = "gpt-3.5-turbo""

In [3]:
# Set up API key
openai.api_key = openai_key  # This was missing

# define function to query openai
def query_openai(prompt, model=chat_model):
    try:
        response = openai.chat.completions.create(
            model=model,  # Now you can specify different models
            messages=[
                {"role": "system", "content": "You are a knowledgeable guide summarizing origins, importance, major battles or wars, and famous people for multiple cities, focusing on salient points. You summary must factor in that the person has cycled through all these towns and you must respond adequantely."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=500,
            temperature=0.7
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"


In [4]:

# Test the function
prompt = "The Hague"
response = query_openai(prompt)
print(textwrap.fill(response, width=100))

The Hague, located in the Netherlands, is known for being the seat of the Dutch government and the
International Court of Justice. It has a rich history dating back to the 13th century when it was
founded as a hunting lodge.  The Hague is an important city for international diplomacy and is home
to many international organizations and embassies. One of the most famous events in the city's
history is the signing of the Treaty of The Hague in 1949, which established NATO.  Some of the
major battles and wars that have involved The Hague include the Dutch War of Independence in the
16th century and World War II when the city was occupied by German forces.  Famous people associated
with The Hague include the Dutch artist Johannes Vermeer, known for his masterpiece "Girl with a
Pearl Earring," and the politician Johan van Oldenbarnevelt, who played a key role in the Dutch
Republic's struggle for independence.  Overall, The Hague is a city with a rich history and a
significant role in interna

In [5]:
route_locations = pd.DataFrame(
    {
        "country": ["The Netherlands", "The Netherlands", "The Netherlands", "The Netherlands", "The Netherlands", "The Netherlands", "The Netherlands", "The Netherlands", "The Netherlands"],
        "towns": ["Segbroek",  "Kraayenstein", "Kwintsheul", "Schipluiden", "Berkel en Rodenrijs", "Pijnacker", "Leidschendam", "Marlot", "Haagse Bos"],
            }
)

In [6]:
route_locations

,country,towns
0,The Netherlands,Segbroek
1,The Netherlands,Kraayenstein
2,The Netherlands,Kwintsheul
3,The Netherlands,Schipluiden
4,The Netherlands,Berkel en Rodenrijs
5,The Netherlands,Pijnacker
6,The Netherlands,Leidschendam
7,The Netherlands,Marlot
8,The Netherlands,Haagse Bos


In [7]:
# Only join unique town names, exclude country
all_unique_towns = route_locations["towns"].unique()
joined_string = ", ".join(map(str, all_unique_towns))
print("All unique towns joined into one string:")
print(textwrap.fill(joined_string, width=80))

All unique towns joined into one string:
Segbroek, Kraayenstein, Kwintsheul, Schipluiden, Berkel en Rodenrijs, Pijnacker,
Leidschendam, Marlot, Haagse Bos


In [8]:

def query_2(city_list, model=chat_model):
    """
    Generate a story about a cyclist's journey through a list of cities.
    city_list: list of city/town names or a comma-separated string.
    """
    if isinstance(city_list, list):
        cities = ", ".join(city_list)
    else:
        cities = city_list

    prompt = (
    f"Retell the cyclist's journey as if this was the dutch golden age, starting and ending at the first of {cities}"
    "weaving a vivid narrative of the route, key historic events, and notable figures tied to the cities."
    "Do not repeat the cities like reading a book. Tell the story as if you reader where actually there!"
)
    try:
        response = openai.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=500,
            temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

In [9]:
print(textwrap.fill(query_2(all_unique_towns), width=80))

Our journey begins in the picturesque town of Segbroek, known for its charming
canals and bustling marketplaces. As we set off on our trusty steed, we pass by
the grand estates of the wealthy merchants who call this town home.  Our next
stop is Kraayenstein, a bustling hub of trade and commerce. Here, we encounter
the famous painter Johannes Vermeer, who is hard at work on his latest
masterpiece. We pause to admire his skill before continuing on our way.  Next,
we arrive in Kwintsheul, a quaint village known for its lush green fields and
bountiful orchards. We stop to sample some of the delicious fruits grown here
before continuing our journey.  Our path takes us through the historic town of
Schipluiden, where we witness a lively market filled with vendors selling their
wares. We stop to chat with some of the locals, who regale us with tales of the
town's rich history.  As we make our way through Berkel en Rodenrijs, we come
across the famous philosopher Baruch Spinoza, who is deep in 

In [10]:
import requests

# Replace with your Strava API access token
access_token = "578aad20e6166cb474ca76e16e2e1d39c7ec2edb"

# Replace with the activity ID you want to fetch
activity_id = 15310399955

In [11]:

def get_strava_activity(activity_id, access_token):
    """
    Fetches details for a specific Strava activity.
    """
    url = f"https://www.strava.com/api/v3/activities/{activity_id}"
    headers = {"Authorization": f"Bearer {access_token}"}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

# Example usage:
activity = get_strava_activity(activity_id, access_token)
if activity:
    print("Activity name:", activity.get("name"))
    print("Distance (meters):", activity.get("distance"))
    print("Moving time (seconds):", activity.get("moving_time"))
    print("Start date:", activity.get("start_date"))
    print("Map summary polyline:", activity.get("map", {}).get("summary_polyline"))

Error: 401 - {"message":"Authorization Error","errors":[{"resource":"Application","field":"","code":"invalid"}]}


In [12]:
import stravalib
import polyline
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [13]:
# initialize strava client
client = stravalib.Client(access_token=access_token)

In [14]:
# get activity
activity = client.get_activity(activity_id)

AccessUnauthorized: Unauthorized: Authorization Error: [{'resource': 'Application', 'field': '', 'code': 'invalid'}]